# Evaluación 1

**Integrante 1:** Cristina Morán

**Integrante 2:** Alonso Valderrama

**Integrante 3:** Amasis Guzmán

**Correo Electrónico integrante 1:** cristina.moran2201@alumnos.ubiobio.cl

**Correo Electrónico integrante 2:** alonso.valderrama2201@alumnos.ubiobio.cl

**Correo Electrónico integrante 3:** amasis.guzman2201@alumnos.ubiobio.cl

**Fecha de Creación:** Abril de 2026  
**Versión:** 1.0  

---

## Descripción

Este notebook contiene el desarrollo de la evaluación 1 de la asignatura Inteligencia Artificial de la carrera de Ingeniería Civil en Informática de la Universidad del Bio Bio, sede Concepción.


---

## Requisitos de Software

Este notebook fue desarrollado con Python 3.12. A continuación se listan las bibliotecas necesarias:

- pandas (>=1.1.0)
- matplotlib (3.7.1)
- numpy (2.0.2)


Para verificar la versión instalada ejecutar usando el siguiente comando, usando la librería de la cual quieres saber la versión:

```python
import pandas as pd
print(pd.__version__)
````

#Carga de datos

In [ ]:
!wget https://raw.githubusercontent.com/JaznaLaProfe/datos/master/preparation_data/dataset_churn_dirty.csv

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sb

from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, StandardScaler, OneHotEncoder

In [ ]:
#Carga el set de datos
data = pd.read_csv('dataset_churn_dirty.csv')

#Para mostrar en una tabla los datos del archivo
data.head() 

In [ ]:
# Cantidad de observaciones y columnas
data.shape

In [ ]:
# revisión de valores faltantes y tipos de datos
data.info()

# muestra la cantidad de nulos por columna
data.isnull().sum()

Comentario: Esto quiere decir que los datos del archivo \.csv ingresado poseen 16 columnas y cada una de ellas 21000 valores\.

# Revisión de nulos

De estos datos podemos observar la siguente cantidad de datos nulos:

age : 1036 datos nulos\.

monthly\_income: 1038 datos nulos\.

gender: 1053 datos nulos\.

Comentario respecto a nulos: Se puede apreciar que las 4 columnas con nulos son "age", "monthly\_income", "gender" y "payment\_method", además que la cantidad de datos total de cada columna debiera ser de 21000 entradas\.

payment\_method : 1047 datos nulos\.

In [ ]:
columnas_con_nulos = data.isna().sum()[data.isna().sum() > 0]
porcentaje_nulos = (columnas_con_nulos / data.shape[0]) * 100

resultado = pd.DataFrame({
    "Cantidad Nulos": columnas_con_nulos,
    "Porcentaje Nulos (%)": porcentaje_nulos
}).round(2)

resultado

Observamos que entre estas 4 columnas tan solo el 5% de los datos o menos son nulos\.

In [ ]:
# Funciones para revisión de atípicos
def buscar_atipicos(data : pd.DataFrame, columna : str) -> pd.DataFrame:
  """
  Busca valores atípicos en una columna.
  """
  # Calcular los límites
  Q1 = data[columna].quantile(0.25)
  Q3 = data[columna].quantile(0.75)
  # Calcula rango intercuartilico
  IQR = Q3 - Q1
  limite_inferior = Q1 - 1.5 * IQR 
  limite_superior = Q3 + 1.5 * IQR

  # Filtrar outliers
  return data[(data[columna] < limite_inferior) | (data[columna] > limite_superior)]

def obtener_cantidad_atipicos(data : pd.DataFrame, columnas : np.array) -> dict:
  """
  Obtiene la cantidad de atípicos por cada columna.
  """
  total_atipicos = {}
  for columna in data[columnas]:
    atipicos = buscar_atipicos(data, columna)
    total_atipicos[columna] = atipicos.shape[0]
  return total_atipicos


De esto podemos decir que las columnas "monthly\_income", "num\_logins\_last\_month" y "avg\_session\_time" presentan una alta cantidad de valores atípicos\.

# Revisión de atípicos

In [ ]:
atipicos_por_columna = obtener_cantidad_atipicos(data, data.describe().columns)
atipicos_por_columna

# Existencia de valores atípicos \(outliers\)

Analizando los datos podemos identificar la presencia de valores atípicos en múltiples variables, siendo las más relevantes: age, monthly\_income, num\_logins\_last\_month, avg\_session\_time, support\_tickets, account\_balance y last\_payment\_amount\.
 

In [ ]:
# Columnas numéricas para revisar atípicos
revision_atipicos = [
    'age',
    'monthly_income',
    'tenure_months',
    'num_logins_last_month',
    'avg_session_time',
    'support_tickets',
    'account_balance',
    'last_payment_amount'
]

# Crear subplots (2 filas x 4 columnas porque son 8 variables)
fig, axes = plt.subplots(2, 4, figsize=(20,30))
axes = axes.flatten()

# Graficar boxplots
for i, col in enumerate(revision_atipicos):
    data[col].plot(kind='box', ax=axes[i])
    axes[i].set_title(col)
    axes[i].tick_params(axis="x", labelrotation=45)

# Título general
plt.suptitle("Análisis de valores atípicos", fontsize=16, fontweight="bold")

# Ajuste de diseño
plt.tight_layout()

plt.show()

EXPLICACIÓN DE CADA VARIABLE:

age \(edad\)
\- Mayoría de las edades entre los 30 y 60 años
\- Mediana está cerca de los 40\-50
\- Existe un error o dato poco realista, valor atípico muy alto \(~140 años\)
\- Existen valores bajos, cercanos a 0
Conclusión: Hay outliers evidentes, especialmente un extremo

monthly\_income \(ingreso mensual\)
\- La mayoría de los ingresos están concentrados en un rango bajo\-medio
\- Existen valores atípicos, incluyendo uno muy alto \(~10 millones\)
\- Existen valores negativos o muy bajos
Conclusión: es una variable con alta dispersión y outliers extremos, probablemente necesita limpieza o transformación\.

tenure\_months \(antigüedad en meses\)
\- Los valores están entre 0 y ~120 meses
\- La mediana está cerca de los 60 meses
\- No se observan muchos outliers
Conclusión: Distribución mayoritariamente normal y estable

num\_logins\_last\_month \(número de inicios de sesión\)
\- La mayoría de los usuarios tienen entre 25 y 35 logins
\- Existen outliers tanto Altos \(~50\-55 logins\) y Bajos \(~10\-15 logins\)
Conclusión: Existe variabilidad en el comportamiento de usuarios

avg\_session\_time \(tiempo promedio de sesión\)
\- La mayoría está entre 50 y 80
\- Existen Valores negativos \(posible error\) y un outlier extremo \(~500\)
Conclusión: Existencia de datos inconsistentes \(probablemente errores o registros mal capturados\)

support\_tickets \(tickets de soporte\)
\- La mayoría de los usuarios tiene entre 1 y 3 tickets
\- Algunos valores llegan hasta 7–9 \(outliers\)
Conclusión: Distribución razonable, pero con algunos clientes que requieren mucho soporte

last\_payment\_amount \(último pago\)
\- La mayoría de pagos está entre 10,000 y 25,000\.
\- Hay outliers negativos \(posible error o devolución\) y unos muy altos \(~60,000\+\)
Conclusión: variable con bastante dispersión\.

account\_balance \(saldo de cuenta\)
\- La mayoría está entre 30,000 y 60,000\.
Existen valores negativos \(cuentas en deuda\) y valores muy altos \(~100,000\+\)
Conclusión: Presencia clara de outliers en ambos extremos\.

En resumen , las variables con más outliers son monthly\_income, account\_balance, last\_payment\_amount, avg\_session\_time
Las variables más estables son tenure\_months, support\_tickets
Existen posibles errores de datos \(valores negativos o muy altos\)\.

In [ ]:
data.drop(columns=['customer_id']).describe()
# Muestra las medidas estadísticas, descartando la columna que identifica al cliente.

In [ ]:
# Detectar inconsistencias
inconsistentes = data[
    (data["age"] <= 0) |
    (data["monthly_income"] < 0) |
    (data["tenure_months"] < 0) |
    (data["num_logins_last_month"] < 0) |
    (data["avg_session_time"] < 0) |
    (data["support_tickets"] < 0) |
    (data["account_balance"] < 0) |
    (data["last_payment_amount"] < 0)
]

print("Registros inconsistentes encontrados:")
inconsistentes

In [ ]:
data.describe(include="object")

Comentario: Tabla de inconsistencias totales\. Aquí podemos ver claramente la existencia de datos inconsistentes como en age que tenemos un \-5, monthly\_income = \-100000, avg\_session\_time con un valor negativo\.

# Revisión de inconsistencias

# Revisión de inconsistencias para cualitativas

In [ ]:
data.gender.unique()

Género

Región

In [ ]:
data.region.unique()

Tipo de subscripcion

In [ ]:
data.subscription_type.unique()

In [ ]:
data.payment_method.unique()

Es activo

In [ ]:
data.is_active.unique()

Método de Pago

Dispositivo favorito

In [ ]:
data.preferred_device.unique()

In [ ]:
# Revisa la existencia de duplicados
data.duplicated().sum()

Esto quiere decir que existen 1000 datos duplicados, por lo tanto cuando se realice la limpieza en lugar de 21000 datos solo contaremos con 20000 o menos

# Revisión de duplicados

Gracias a la variable customer\_id podemos identificar los datos duplicados, los cuales confirman la identidad entre datos\. Concluyendo que hay 2000 duplicados\.

In [ ]:
data.describe(include="object")

# Limpieza y transformación

Diferenciamos las variables cualitativas de las cuantitativas agrupándolas en dos arreglos diferentes para manejarlas de forma correcta más adelante\.

In [ ]:
# Obtiene los registos duplicados
data[data.duplicated(keep=False)].sort_values(by='customer_id')

In [ ]:
# Define las variables cualitativas y cuantitativas disponibles
numeric_features = ['age', 'monthly_income', 'tenure_months', 'num_logins_last_month', 'avg_session_time', 'support_tickets', 'account_balance', 'last_payment_amount']
categorical_features = ['gender', 'payment_method', 'preferred_device', 'region', 'subscription_type', 'is_active']

Luego de definir Windsorizer podemos definir los Pipeline tanto para variables cuantitativas como cualitativas

Creamos la clase Windsorizer para tratar los valores atípicos \(outliers\) reemplazando los valores extremos \(muy altos o muy bajos\) por percentiles específicos

En este caso, Winsorizer está actuando con el 5% más bajo y el 5% más alto de los datos atípicos \(percentil 5 y percentil 95\)\. De este modo los valores extremos desaparecen\.

Primero se define la clase con sus funciones para eliminar los valores duplicados

In [ ]:
class Winsorizer(BaseEstimator, TransformerMixin): #le entregas los minimos o utiliza los default
  """
  Tratamiento de atípicos

  Parámetros
  ----------
  BaseEstimator : Clase base para estimadores en scikit-learn.
  TransformerMixin : Clase base para transformadores en scikit-learn.

  Atributos
  ---------
  columns_ : array-like
    Nombres de las columnas a transformar.
  limits : tuple
    % de los extremos a descartar
  """
  def __init__(self, limits=(0.05, 0.05)):
    self.limits = limits

  def fit(self, X, y=None):
    # Guardar nombres si es DataFrame,sino generar nombres genéricos
    if isinstance(X, pd.DataFrame):
      self.columns_ = X.columns
    else:
      self.columns_ = np.arange(X.shape[1])
    return self

  def transform(self, X):
    X = pd.DataFrame(X, columns=self.columns_)
    for col in self.columns_:
      lower = X[col].quantile(self.limits[0])
      upper = X[col].quantile(1 - self.limits[1])
      X = X.astype("float64")
      X[col] = np.clip(X[col], lower, upper)
    return X

  def get_feature_names_out(self, input_features=None):
    if input_features is None:
      return np.array(self.columns_)
    else:
      return np.array(input_features)

## Pipeline Variables Cuantitativas:

In [ ]:
# Eliminamos los valores duplicados
class DropDuplicatesTransformer(BaseEstimator, TransformerMixin):
    """
    Elimina filas duplicadas dentro del pipeline, dejando solo una.

    Parámetros
    ----------
    BaseEstimator : Clase base para estimadores en scikit-learn.
    TransformerMixin : Clase base para transformadores en scikit-learn.
    """
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            return X.drop_duplicates().reset_index(drop=True)
        return X


Variables numéricas \(age, monthly\_income, tenure\_months, num\_logins\_last\_month, avg\_session\_time, support\_tickets, account\_balance, last\_payment\_amount\) → Se imputan con la media

In [ ]:
# Define el pipeline para las variables cuantitativas
pipeline_numerico = Pipeline(
    steps=[
        ("winsorizer", Winsorizer()),
        ("imputacion", SimpleImputer(strategy="mean")), # Detecta Nulos y reemplaza por media
        ("escalado", StandardScaler()) # Varianza
    ]
)

## Pipeline Variables Cualitativas:

Y ahora que ya están definidas las funciones para el tratamiento de las variables cuantitativas podemos integrarlas al pipeline numerico

Ahora procedemos a definir el pipeline para las variables cualitativas con sus funciones correspondientes

In [ ]:
# Define el pipeline para las variables cualitativas
pipeline_categorico = Pipeline(steps=[
    ("imputacion", SimpleImputer(strategy="most_frequent")),
    ("codificacion", OneHotEncoder(handle_unknown="ignore"))
])
#Si hay valores Nulos los reemplazamos por el valor mas repetido osea la Moda

Integramos ambos pipeline

In [ ]:
# Integra ambos pipelines
preprocesador = ColumnTransformer(
    transformers=[
        ("num", pipeline_numerico, numeric_features),
        ("cat", pipeline_categorico, categorical_features)
    ]
)
# Tratamiento duplicados
pipeline_completo = Pipeline(
    steps=[
        ("eliminar_duplicados", DropDuplicatesTransformer()),
        ("preprocesador", preprocesador)
    ]
)
#Eliminamos duplicados y limpiamos categorias cuantitativas.

In [ ]:
data.info()

Variables categóricas \(gender, payment\_method, preferred\_device, region, subscription\_type, is\_active\) \-\> Se imputan con la moda

In [ ]:
# Estandariza los strings mal escritos (corrige mayúsculas y minúsculas)
data["gender"] = data["gender"].str.strip().str.capitalize()
data["payment_method"] = data["payment_method"].str.strip().str.lower()
data["preferred_device"] = data["preferred_device"].str.strip().str.lower()
data["region"] = data["region"].str.strip().str.lower()
data["subscription_type"] = data["subscription_type"].str.strip().str.lower()
data["is_active"] = data["is_active"].str.strip().str.lower()

#Estandarizamos codigo para que los datos sean eficientes y de calidad.

In [ ]:
# Limpia las inconsistencias encontradas
data.loc[data["age"]<= 0, "age"] = np.nan
data.loc[data["age"]<= 120, "age"] = np.nan
data.loc[data["monthly_income"]<= 0, "monthly_income"] = np.nan
data.loc[data["avg_session_time"]<= 0, "avg_session_time"] = np.nan
data.loc[data["account_balance"]<= 0, "account_balance"] = np.nan
data.loc[data["last_payment_amount"]<= 0, "last_payment_amount"] = np.nan
data.loc[data["tenure_months"]<= 0, "tenure_months"] = np.nan
data.loc[data["support_tickets"]<= 0, "support_tickets"] = np.nan
data.loc[data["num_logins_last_month"]<= 0, "num_logins_last_month"] = np.nan
data.shape[0]

#Reemplaza inconsistencias con un nan

In [ ]:
# Aplicar el pipeline completo
X = data.drop(columns=["churn", "customer_id"])
y = data["churn"]
data_transformada = pipeline_completo.fit_transform(X)
#Con esto separamos,limpiamos y transformamos los datos para que esten de calidad.

# Guarda set de datos limpio y transformado

In [ ]:
# Feature Engineering
data["login_rate"] = data["num_logins_last_month"] / (data["tenure_months"] + 1)
data["income_to_payment_ratio"] = data["last_payment_amount"] / (data["monthly_income"] + 1)

#Creamos una nueva columna que muestra cuantas veces se logea un user 
#por mes desde que esta activo


print("Variables creadas:")
print(data[["login_rate", "income_to_payment_ratio"]].describe())

In [ ]:
# guardar la matriz transformada como DataFrame
columnas_out = preprocesador.get_feature_names_out()
data_final = pd.DataFrame(data_transformada, columns=columnas_out)
y_aligned = data.drop_duplicates(subset=X.columns).reset_index(drop=True)["churn"]

# Agregar la variable objetivo para que el dataset quede listo para modelado
data_final["churn"] = y_aligned.values
data_final.to_csv("dataset_churn_transformado.csv", index=False)

In [ ]:
pd.DataFrame(data_transformada, columns=columnas_out).info()

Finalmente, el proceso de preparación aplicado al dataset de churn permitió transformar 21\.000 registros y 16 columnas con múltiples problemas de calidad \(valores nulos, filas duplicadas, outliers extremos y datos inconsistentes\) en un dataset limpio y listo para modelado, con 20000 datos no nulos\. 

Se diseñó un pipeline estructurado y reproducible en scikit\-learn que integró: eliminación de duplicados, winsorización, imputación, estandarización y codificación One\-Hot de variables categóricas\. Además, se crearon dos nuevas variables derivadas \(login\_rate e income\_to\_payment\_ratio\) que servirán para aportar valor predictivo en etapas posteriores\. 

El resultado fue un dataset sin nulos ni duplicados, correctamente transformado y exportado como dataset\_churn\_transformado\.csv\. 

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=412dc4ce-25a6-448b-a3d9-1501e633b989' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>